<a href="https://colab.research.google.com/github/Zsombroo/Machine-Learning-Defense-Techniques/blob/master/Defensive_Distillation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Defensive distillation

This is an example project of how to implement defensive distillation using Keras

https://arxiv.org/abs/1511.04508

In [0]:
# Install missing packages
!pip install cleverhans

     |████████████████████████████████| 204kB 5.2MB/s 
     |████████████████████████████████| 51kB 17.4MB/s 


In [0]:
# Import dependencies
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from cleverhans.attacks import FastGradientMethod
from cleverhans.dataset import MNIST
from cleverhans.utils_keras import KerasModelWrapper

from keras import backend as K
from keras.callbacks import EarlyStopping
from keras.callbacks import ModelCheckpoint
from keras.layers import Activation
from keras.layers import Convolution2D
from keras.layers import Dense
from keras.layers import Dropout
from keras.layers import Flatten
from keras.layers import Input
from keras.layers import Lambda
from keras.layers import MaxPooling2D
from keras.models import load_model
from keras.models import Model

tf.set_random_seed(100)
np.random.seed(100)
K.set_learning_phase(0)
sess = tf.Session()
K.set_session(sess)

W0827 14:30:09.998782 140061849581440 deprecation_wrapper.py:119] From /usr/local/lib/python3.6/dist-packages/cleverhans/utils_tf.py:341: The name tf.GraphKeys is deprecated. Please use tf.compat.v1.GraphKeys instead.

Using TensorFlow backend.
W0827 14:30:10.133115 140061849581440 deprecation_wrapper.py:119] From /usr/local/lib/python3.6/dist-packages/keras/backend/tensorflow_backend.py:153: The name tf.get_default_graph is deprecated. Please use tf.compat.v1.get_default_graph instead.



### Prepare data

In [0]:
# Load MNIST dataset
mnist = MNIST(train_start=0, train_end=60000, test_start=0, test_end=10000)
x_train, y_train = mnist.get_set('train')
x_test, y_test = mnist.get_set('test')

x_train_1 = x_train[:30000]
x_train_2 = x_train[30000:]
y_train_1 = y_train[:30000]
y_train_2 = y_train[30000:]

x_test_1 = x_test[:5000]
x_test_2 = x_test[5000:]
y_test_1 = y_test[:5000]
y_test_2 = y_test[5000:]

### Create first model

In [0]:
# Create convolutional neural network
input_layer = Input(shape=(28, 28, 1))
x = Convolution2D(20, (3, 3), activation='relu')(input_layer)
x = MaxPooling2D((2, 2))(x)
x = Convolution2D(20, (3, 3), activation='relu')(x)
x = MaxPooling2D((2, 2))(x)
x = Convolution2D(10, (3, 3), activation='relu')(x)
x = MaxPooling2D((2, 2))(x)
x = Flatten()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.2)(x)
x = Dense(64, activation='relu')(x)
x = Dropout(0.2)(x)
x = Dense(10)(x)
x = Lambda(lambda x: x/50)(x)
output_layer = Activation('softmax')(x)

first_model = Model(inputs=input_layer, outputs=output_layer)
first_model.summary()
first_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

W0827 14:30:14.525949 140061849581440 deprecation_wrapper.py:119] From /usr/local/lib/python3.6/dist-packages/keras/backend/tensorflow_backend.py:517: The name tf.placeholder is deprecated. Please use tf.compat.v1.placeholder instead.

W0827 14:30:14.538539 140061849581440 deprecation_wrapper.py:119] From /usr/local/lib/python3.6/dist-packages/keras/backend/tensorflow_backend.py:4138: The name tf.random_uniform is deprecated. Please use tf.random.uniform instead.

W0827 14:30:14.570557 140061849581440 deprecation_wrapper.py:119] From /usr/local/lib/python3.6/dist-packages/keras/backend/tensorflow_backend.py:3976: The name tf.nn.max_pool is deprecated. Please use tf.nn.max_pool2d instead.

W0827 14:30:14.664669 140061849581440 deprecation_wrapper.py:119] From /usr/local/lib/python3.6/dist-packages/keras/optimizers.py:790: The name tf.train.Optimizer is deprecated. Please use tf.compat.v1.train.Optimizer instead.

W0827 14:30:14.691270 140061849581440 deprecation_wrapper.py:119] From /us

_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_1 (InputLayer)         (None, 28, 28, 1)         0         
_________________________________________________________________
conv2d_1 (Conv2D)            (None, 26, 26, 20)        200       
_________________________________________________________________
max_pooling2d_1 (MaxPooling2 (None, 13, 13, 20)        0         
_________________________________________________________________
conv2d_2 (Conv2D)            (None, 11, 11, 20)        3620      
_________________________________________________________________
max_pooling2d_2 (MaxPooling2 (None, 5, 5, 20)          0         
_________________________________________________________________
conv2d_3 (Conv2D)            (None, 3, 3, 10)          1810      
_________________________________________________________________
max_pooling2d_3 (MaxPooling2 (None, 1, 1, 10)          0         
__________

### Train first model

In [0]:
# Train model
callbacks = [
    EarlyStopping(monitor='val_loss', min_delta=0.001, patience=10, verbose=1, mode='min'),
    ModelCheckpoint('weights.hdf5', monitor='val_loss', verbose=0, save_best_only=True, mode='min')
]

first_model.fit(x_train_1, y_train_1, batch_size=64, epochs=2000, verbose=0, callbacks=callbacks, validation_split=0.1)

W0827 14:30:14.848087 140061849581440 deprecation.py:323] From /usr/local/lib/python3.6/dist-packages/tensorflow/python/ops/math_grad.py:1250: add_dispatch_support.<locals>.wrapper (from tensorflow.python.ops.array_ops) is deprecated and will be removed in a future version.
Instructions for updating:
Use tf.where in 2.0, which has the same broadcast rule as np.where


Epoch 00053: early stopping


In [0]:
# Load best model
first_model = load_model('weights.hdf5')

In [0]:
# Test model accuracy
evaluation = first_model.evaluate(x_test, y_test, verbose=1)
print('Loss:', evaluation[0])
print('Accuracy', evaluation[1])

10000/10000 [==============================] - 1s 91us/step
Loss: 0.10320193754211068
Accuracy 0.9708


### Change hard labels to soft labels

In [0]:
y_train_2 = first_model.predict(x_train_2)

### Create distilled model

In [0]:
# Create convolutional neural network
input_layer = Input(shape=(28, 28, 1))
x = Convolution2D(20, (3, 3), activation='relu')(input_layer)
x = MaxPooling2D((2, 2))(x)
x = Convolution2D(20, (3, 3), activation='relu')(x)
x = MaxPooling2D((2, 2))(x)
x = Convolution2D(10, (3, 3), activation='relu')(x)
x = MaxPooling2D((2, 2))(x)
x = Flatten()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.2)(x)
x = Dense(64, activation='relu')(x)
x = Dropout(0.2)(x)
x = Dense(10)(x)
x = Lambda(lambda x: x/50)(x)
output_layer = Activation('softmax')(x)

distilled_model = Model(inputs=input_layer, outputs=output_layer)
distilled_model.summary()
distilled_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_2 (InputLayer)         (None, 28, 28, 1)         0         
_________________________________________________________________
conv2d_4 (Conv2D)            (None, 26, 26, 20)        200       
_________________________________________________________________
max_pooling2d_4 (MaxPooling2 (None, 13, 13, 20)        0         
_________________________________________________________________
conv2d_5 (Conv2D)            (None, 11, 11, 20)        3620      
_________________________________________________________________
max_pooling2d_5 (MaxPooling2 (None, 5, 5, 20)          0         
_________________________________________________________________
conv2d_6 (Conv2D)            (None, 3, 3, 10)          1810      
_________________________________________________________________
max_pooling2d_6 (MaxPooling2 (None, 1, 1, 10)          0         
__________

### Train distilled model

In [0]:
# Train model
callbacks = [
    EarlyStopping(monitor='val_loss', min_delta=0.001, patience=10, verbose=1, mode='min'),
    ModelCheckpoint('weights.hdf5', monitor='val_loss', verbose=0, save_best_only=True, mode='min')
]

distilled_model.fit(x_train_1, y_train_1, batch_size=64, epochs=2000, verbose=0, callbacks=callbacks, validation_split=0.1)

Epoch 00054: early stopping


In [0]:
# Load best model
distilled_model = load_model('weights.hdf5')

### Reset temperature

In [0]:
input_layer = Input(shape=(28, 28, 1))
x = Convolution2D(20, (3, 3), activation='relu')(input_layer)
x = MaxPooling2D((2, 2))(x)
x = Convolution2D(20, (3, 3), activation='relu')(x)
x = MaxPooling2D((2, 2))(x)
x = Convolution2D(10, (3, 3), activation='relu')(x)
x = MaxPooling2D((2, 2))(x)
x = Flatten()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.2)(x)
x = Dense(64, activation='relu')(x)
x = Dropout(0.2)(x)
x = Dense(10)(x)
output_layer = Activation('softmax')(x)

model = Model(inputs=input_layer, outputs=output_layer)

model.layers[0].set_weights(distilled_model.layers[0].get_weights())
model.layers[1].set_weights(distilled_model.layers[1].get_weights())
model.layers[2].set_weights(distilled_model.layers[2].get_weights())
model.layers[3].set_weights(distilled_model.layers[3].get_weights())
model.layers[4].set_weights(distilled_model.layers[4].get_weights())
model.layers[5].set_weights(distilled_model.layers[5].get_weights())
model.layers[6].set_weights(distilled_model.layers[6].get_weights())
model.layers[7].set_weights(distilled_model.layers[7].get_weights())
model.layers[8].set_weights(distilled_model.layers[8].get_weights())
model.layers[9].set_weights(distilled_model.layers[9].get_weights())
model.layers[10].set_weights(distilled_model.layers[10].get_weights())
model.layers[11].set_weights(distilled_model.layers[11].get_weights())
model.layers[12].set_weights(distilled_model.layers[12].get_weights())
model.layers[13].set_weights(distilled_model.layers[14].get_weights())

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [0]:
# Test model accuracy
evaluation = model.evaluate(x_test, y_test, verbose=1)
print('Loss:', evaluation[0])
print('Accuracy', evaluation[1])

10000/10000 [==============================] - 1s 121us/step
Loss: 0.3953322275947312
Accuracy 0.974


### Generate adversarial examples

In [0]:
fgsm_parameters = {
    'eps': 0.1,
    'ord': np.inf,
    'clip_min': 0.,
    'clip_max': 1.
}

wrap = KerasModelWrapper(model)
fgsm = FastGradientMethod(wrap, sess=sess)
x_adv = fgsm.generate_np(x_test, **fgsm_parameters)

[INFO 2019-08-27 14:38:31,696 cleverhans] Constructing new graph for attack FastGradientMethod
I0827 14:38:31.696066 140061849581440 __init__.py:130] Constructing new graph for attack FastGradientMethod
W0827 14:38:31.746828 140061849581440 deprecation.py:323] From /usr/local/lib/python3.6/dist-packages/cleverhans/attacks/__init__.py:283: to_float (from tensorflow.python.ops.math_ops) is deprecated and will be removed in a future version.
Instructions for updating:
Use `tf.cast` instead.
W0827 14:38:31.788246 140061849581440 deprecation.py:506] From /usr/local/lib/python3.6/dist-packages/cleverhans/compat.py:124: calling softmax_cross_entropy_with_logits_v2_helper (from tensorflow.python.ops.nn_ops) with dim is deprecated and will be removed in a future version.
Instructions for updating:
dim is deprecated, use axis instead


In [0]:
# Test model accuracy
evaluation = model.evaluate(x_adv, y_test, verbose=1)
print('Loss:', evaluation[0])
print('Accuracy', evaluation[1])

10000/10000 [==============================] - 1s 76us/step
Loss: 0.5670166146755239
Accuracy 0.9647
